# Open-Ended MedVQA Benchmark (BLEU / ROUGE / BERTScore) -- Kaggle version

This is a **standalone notebook** for the open-ended questions (`open.csv`),
separate from your closed-question Yes/No pipeline. It runs on **Kaggle**
instead of Colab.

**Before running, in the Kaggle notebook editor (right-hand panel):**
1. **Add Data** -> upload/attach your dataset zip (e.g. `Neck_Test_with_CoT.zip`,
   `BrainFace_Test_with_CoT.zip`). Kaggle auto-extracts zipped datasets, so
   they'll appear under `/kaggle/input/<your-dataset-slug>/...`.
2. **Settings -> Accelerator -> GPU T4 x2** (or P100 if that's what's offered).
   T4 x2 gives you ~30GB of VRAM combined via `device_map="auto"`, which is
   more forgiving than Colab's single T4.
3. **Settings -> Internet -> On** (required to `pip install` and to download
   model weights from Hugging Face).
4. **Add-ons -> Secrets -> add a secret named `HF_TOKEN`** with a Hugging
   Face access token that has accepted the license for the gated Gemma/
   MedGemma repos on huggingface.co (visit each model page once and click
   "Agree and access repository"). This notebook reads that secret below.

**Model choices (same reasoning as the closed-question notebook):**
- `google/gemma-4-E4B-it` instead of the 31B Gemma 4 (31B needs ~18-20GB even
  in 4-bit -- too big for a single T4; the E4B variant fits easily, and with
  T4 x2 here you have even more headroom).
- `google/medgemma-4b-it` added as a medically fine-tuned model -- a good bet
  to beat the general-purpose VLMs on this MedVQA task, and the cheapest of
  the five to run.

## 0. Install dependencies

In [1]:
!pip install -q -U bitsandbytes
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U accelerate
!pip install -q "pillow<11"
!pip install -q rouge-score bert-score nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 34.2 MB/s eta 0:00:0000:0100:01m
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 9.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 67.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 7.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 45.2 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.5 MB/s eta 0:00:00


In [2]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

In [3]:
import os
import gc
import json
import glob
import zipfile
import torch
import pandas as pd
from PIL import Image

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bertscore_score

_smoothing = SmoothingFunction().method1
_rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

## 1. Hugging Face login (needed for gated Gemma / MedGemma repos)

In [4]:
# ---------------------------------------------------------------------------
# Reads the HF_TOKEN secret you added in Add-ons -> Secrets, and logs in so
# gated repos (Gemma 4, MedGemma) can be downloaded. If you haven't added the
# secret yet, this cell will raise -- go add it first (see intro cell above).
# ---------------------------------------------------------------------------
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)
print("Logged in to Hugging Face.")

Logged in to Hugging Face.


## 2. Choose dataset

Set `DATASET` to `"Neck"` or `"BrainFace"`. Unlike Colab, you don't need to
restart anything between runs on a "session" boundary the same way -- but it's
still cleanest to run one dataset fully, save its outputs, then **Restart
Session** (top menu) and switch `DATASET` before running the other, so GPU
memory and disk from the first run are fully released.

In [5]:
# ---------------------------------------------------------------------------
# 1. Choose which dataset to run this session.
#    KAGGLE_INPUT_DIRS: the folder name Kaggle mounts your dataset under is
#    whatever slug it derived from your dataset's title -- check the panel
#    on the right ("Data") or run `os.listdir("/kaggle/input")` below to
#    confirm the exact names, then edit these two paths to match.
# ---------------------------------------------------------------------------
DATASET = "BrainFace"   # <-- change to "BrainFace" for the second run

assert DATASET in ("Neck", "BrainFace"), "DATASET must be 'Neck' or 'BrainFace'"

#print("Available Kaggle input datasets:", os.listdir("/kaggle/input"))

KAGGLE_INPUT_DIRS = {
    "Neck": "/kaggle/input/datasets/arups330/slackdataset/Neck_Test_with_CoT/Neck_Test_with_CoT",
    "BrainFace": "/kaggle/input/datasets/arups330/slackdataset/BrainFace_Test_with_CoT/BrainFace_Test_with_CoT",
}

input_dir = KAGGLE_INPUT_DIRS[DATASET]

# Kaggle auto-extracts zip datasets, so input_dir should already contain the
# unzipped contents. If for some reason you attached the raw .zip itself
# (input_dir points straight at a .zip file), extract it into /kaggle/working:
if os.path.isfile(input_dir) and input_dir.endswith(".zip"):
    extract_dir = f"/kaggle/working/{DATASET}_Test_with_CoT"
    if not os.path.exists(extract_dir):
        print(f"Extracting {input_dir}...")
        with zipfile.ZipFile(input_dir, "r") as zf:
            zf.extractall(extract_dir)
    TEST_DATA_ROOT = extract_dir
else:
    TEST_DATA_ROOT = input_dir

print("\nTEST_DATA_ROOT =", TEST_DATA_ROOT)
print(f"Contents of {TEST_DATA_ROOT}:")
print(" ", os.listdir(TEST_DATA_ROOT))


TEST_DATA_ROOT = /kaggle/input/datasets/arups330/slackdataset/BrainFace_Test_with_CoT/BrainFace_Test_with_CoT
Contents of /kaggle/input/datasets/arups330/slackdataset/BrainFace_Test_with_CoT/BrainFace_Test_with_CoT:
  ['CT']


In [6]:
MAX_ROWS_PER_MODEL = None   # set e.g. 30 for a quick test run before the full set

# All Kaggle outputs go under /kaggle/working/ -- that's the only writable,
# persisted-to-notebook-output directory (equivalent of Colab's /content/).
OUTPUT_DIR = "/kaggle/working"

## 3. Load `open.csv` for the selected dataset

In [7]:
# ---------------------------------------------------------------------------
# 2. Load open-ended test data for the selected dataset only
#    (with the "CoT" column already filled in)
# ---------------------------------------------------------------------------
open_csvs = glob.glob(os.path.join(TEST_DATA_ROOT, "**", "open.csv"), recursive=True)

print(f"Found {len(open_csvs)} open.csv test files for {DATASET}:")
for p in open_csvs:
    print(" ", p)

open_frames = []
for csv_path in open_csvs:
    split_dir = os.path.dirname(csv_path)
    df = pd.read_csv(csv_path)
    df["split_dir"] = split_dir
    open_frames.append(df)

open_df = pd.concat(open_frames, ignore_index=True)
open_df = open_df[open_df["CoT"].notna() & (open_df["CoT"].str.strip() != "")]
if MAX_ROWS_PER_MODEL:
    open_df = open_df.head(MAX_ROWS_PER_MODEL)
print(f"\nTotal open-ended test rows for {DATASET} (with valid CoT): {len(open_df)}")

OPEN_IMG_COL = "image_file" if "image_file" in open_df.columns else "img_name"

def resolve_image_path(split_dir: str, img_name: str) -> str:
    flat_name = os.path.basename(str(img_name))
    for c in [os.path.join(split_dir, str(img_name)), os.path.join(split_dir, flat_name),
              os.path.join(split_dir, str(img_name).replace("/", "_"))]:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"Could not find image for {img_name!r} in {split_dir}")

Found 1 open.csv test files for BrainFace:
  /kaggle/input/datasets/arups330/slackdataset/BrainFace_Test_with_CoT/BrainFace_Test_with_CoT/CT/test/open.csv

Total open-ended test rows for BrainFace (with valid CoT): 17


## 4. System prompt (your latest version, 1-4 word free-text answers)

In [8]:
# ---------------------------------------------------------------------------
# 3. System prompt for open-ended questions
# ---------------------------------------------------------------------------
def systemPrompt(question, cot):
    return f"""
              Context:
                     You are a board-certified radiologist and Medical Visual Question Answering (MedVQA) expert with experience interpreting X-ray, CT, MRI, Ultrasound, and other medical images.

              Objective:
                      Answer the user's question by verifying whether it is supported by the visual evidence in the medical image.

              Inputs:
              Question: {question}

              You have to think step by step.

              Instructions:
              1. Examine the medical image carefully.
              2. Independently determine the relevant visual findings before considering the CoT.
              3. Compare your own observations with the provided CoT.
              4. If the CoT is inconsistent with the image, disregard it.
              5. Answer the question using the following evidence priority:
                1. Medical image (highest priority)
                2. User question
                3. CoT (only if verified by the image)
              6. Never fabricate findings or rely on assumptions.

              Output Requirements:
              - Return ONLY the final answer in 1 to 4 words. Do not explain.
              Reasoning Reference:
              {cot}

              Use it only if it agrees with the image.
            """

## 5. Model registry

`google/gemma-4-E4B-it` (fits well even on a single T4, and Kaggle's T4 x2
gives extra headroom) and `google/medgemma-4b-it` as the medically-tuned
candidate. Comment out any model in `MODEL_REGISTRY` you don't want to run.

In [9]:
# ---------------------------------------------------------------------------
# 4. Model registry -- repo id + loader + free-text generation.
# ---------------------------------------------------------------------------
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

def _load_generic(repo, token=None):
    processor = AutoProcessor.from_pretrained(repo, token=token or HF_TOKEN)
    model = AutoModelForImageTextToText.from_pretrained(
        repo, quantization_config=bnb_config, device_map="auto", token=token or HF_TOKEN,
    )
    return model, processor

def load_llama_vision():
    # Unsloth's ungated public mirror of Llama-3.2-11B-Vision-Instruct --
    # avoids needing to separately accept Meta's license.
    return _load_generic("unsloth/Llama-3.2-11B-Vision-Instruct")

def load_qwen25_vl():
    return _load_generic("Qwen/Qwen2.5-VL-7B-Instruct")

def load_qwen3_vl():
    return _load_generic("Qwen/Qwen3-VL-8B-Instruct")

def load_gemma4():
    # google/gemma-4-31B-it (dense 31B) needs ~18-20GB even in 4-bit --
    # doesn't reliably fit a single T4. This smaller multimodal Gemma 4
    # variant fits comfortably.
    return _load_generic("google/gemma-4-E4B-it")

def load_medgemma():
    # Google's Gemma-3-based model fine-tuned specifically on medical images
    # and medical VQA data -- a strong candidate on this task, and the
    # cheapest model here (4B) to run.
    return _load_generic("google/medgemma-4b-it")

def generate_free_text(model, processor, image, prompt_text, max_new_tokens=16):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt_text},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)

    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return processor.decode(generated, skip_special_tokens=True).strip()

MODEL_REGISTRY = {
    "Gemma4-E4B":    load_gemma4,
    "MedGemma-4B":   load_medgemma,
    "Qwen2.5-VL-7B": load_qwen25_vl,
    "Llama-11B":     load_llama_vision,
    "Qwen3-VL-8B":   load_qwen3_vl,
}

REPO_IDS = {
    "Gemma4-E4B": "google/gemma-4-E4B-it",
    "MedGemma-4B": "google/medgemma-4b-it",
    "Qwen2.5-VL-7B": "Qwen/Qwen2.5-VL-7B-Instruct",
    "Llama-11B": "unsloth/Llama-3.2-11B-Vision-Instruct",
    "Qwen3-VL-8B": "Qwen/Qwen3-VL-8B-Instruct",
}

## 6. Run one model/condition over `open_df`

In [10]:
# ---------------------------------------------------------------------------
# 5. Run one model, one condition (with/without CoT), over open_df
# ---------------------------------------------------------------------------
def run_condition_open(model, processor, use_cot: bool):
    results = []
    for i, row in open_df.iterrows():
        try:
            img_path = resolve_image_path(row["split_dir"], row[OPEN_IMG_COL])
            image = Image.open(img_path).convert("RGB")
            cot_text = row["CoT"] if use_cot else ""
            prompt_text = systemPrompt(row["question"], cot_text)
            raw_output = generate_free_text(model, processor, image, prompt_text)
        except Exception as e:
            print(f"    [WARN] row {i} failed: {e}")
            raw_output = ""

        gold = str(row["answer"]).strip()
        results.append({"question": row["question"], "gold": gold, "pred": raw_output})

        if i % 20 == 0:
            print(f"    [{i}/{len(open_df)}] gold={gold!r} pred={raw_output!r}")
    return results

## 7. Generation-quality metrics: BLEU, ROUGE-1/2/L, BERTScore P/R/F1

In [11]:
# ---------------------------------------------------------------------------
# 6. Compute BLEU / ROUGE-1 / ROUGE-2 / ROUGE-L / BERTScore P,R,F1
#    over a full set of (gold, pred) pairs for one model+condition.
# ---------------------------------------------------------------------------
def compute_generation_metrics(golds, preds):
    # --- BLEU (sentence-level, averaged) ---
    bleu_scores = []
    for g, p in zip(golds, preds):
        ref_tokens = [g.lower().split()]
        hyp_tokens = p.lower().split()
        if len(hyp_tokens) == 0:
            bleu_scores.append(0.0)
            continue
        bleu_scores.append(sentence_bleu(ref_tokens, hyp_tokens, smoothing_function=_smoothing))
    bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0.0

    # --- ROUGE-1 / ROUGE-2 / ROUGE-L (f-measure, averaged) ---
    r1, r2, rl = [], [], []
    for g, p in zip(golds, preds):
        scores = _rouge.score(g, p)
        r1.append(scores["rouge1"].fmeasure)
        r2.append(scores["rouge2"].fmeasure)
        rl.append(scores["rougeL"].fmeasure)
    rouge1 = sum(r1) / len(r1) if r1 else 0.0
    rouge2 = sum(r2) / len(r2) if r2 else 0.0
    rougeL = sum(rl) / len(rl) if rl else 0.0

    # --- BERTScore (P, R, F1) ---
    P, R, F1 = bertscore_score(preds, golds, lang="en", verbose=False)
    bert_p = P.mean().item()
    bert_r = R.mean().item()
    bert_f1 = F1.mean().item()

    return {
        "BLEU": round(bleu, 4),
        "ROUGE-1": round(rouge1, 4),
        "ROUGE-2": round(rouge2, 4),
        "ROUGE-L": round(rougeL, 4),
        "BERTScore F1": round(bert_f1, 4),
        "BERTScore P": round(bert_p, 4),
        "BERTScore R": round(bert_r, 4),
    }

In [12]:
# ---------------------------------------------------------------------------
# 7. Disk-cache cleanup between models (Kaggle's /kaggle/working quota is
#    limited too, and the HF cache defaults to ~/.cache which counts against
#    your session's disk).
# ---------------------------------------------------------------------------
import shutil

def clear_model_cache(repo_id: str):
    """Delete this repo's downloaded weights from the local HF cache to
    free disk space before the next model loads."""
    cache_dir = os.path.expanduser("~/.cache/huggingface/hub")
    folder_name = "models--" + repo_id.replace("/", "--")
    path = os.path.join(cache_dir, folder_name)
    if os.path.exists(path):
        size_gb = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, _, files in os.walk(path) for f in files
        ) / (1024**3)
        shutil.rmtree(path, ignore_errors=True)
        print(f"Cleared cache for {repo_id} (freed ~{size_gb:.1f} GB)")
    else:
        print(f"No cache found for {repo_id} (nothing to clear)")

## 8. Run all models x both CoT conditions, save results to `/kaggle/working`

In [13]:
# ---------------------------------------------------------------------------
# 8. Run ALL models x BOTH conditions on the open-ended set, computing
#    generation-quality metrics -- DATASET-specific filenames, saved to
#    /kaggle/working so they show up in the notebook's Output tab.
# ---------------------------------------------------------------------------
open_summary_rows = []
all_open_results = {}

for model_name, loader_fn in MODEL_REGISTRY.items():
    print(f"\n{'='*70}\nLoading {model_name}  [{DATASET} - OPEN]\n{'='*70}")
    try:
        model, processor = loader_fn()
    except Exception as e:
        print(f"  [SKIPPING {model_name}] failed to load: {e}")
        clear_model_cache(REPO_IDS[model_name])
        continue

    try:
        for condition, use_cot in [("With", True), ("Without", False)]:
            print(f"\n--- {model_name} | CoT: {condition} | Dataset: {DATASET} | OPEN ---")
            results = run_condition_open(model, processor, use_cot)
            all_open_results[f"{model_name}_{condition}"] = results

            golds = [r["gold"] for r in results]
            preds = [r["pred"] for r in results]

            metrics = compute_generation_metrics(golds, preds)
            print(f"Metrics -- {model_name} ({condition} CoT, {DATASET}): {metrics}")

            open_summary_rows.append({
                "Dataset": DATASET,
                "Model Name": model_name,
                "CoT": condition,
                **metrics,
            })

            pd.DataFrame(open_summary_rows).to_csv(
                f"{OUTPUT_DIR}/open_comparison_summary_{DATASET}.csv", index=False
            )
            with open(f"{OUTPUT_DIR}/open_comparison_raw_results_{DATASET}.json", "w") as f:
                json.dump(all_open_results, f, indent=2)

    finally:
        del model, processor
        gc.collect()
        torch.cuda.empty_cache()
        clear_model_cache(REPO_IDS[model_name])


Loading Gemma4-E4B  [BrainFace - OPEN]


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Metrics -- Qwen3-VL-8B (With CoT, BrainFace): {'BLEU': 0.1985, 'ROUGE-1': 0.8007, 'ROUGE-2': 0.1307, 'ROUGE-L': 0.8007, 'BERTScore F1': 0.9662, 'BERTScore P': 0.9656, 'BERTScore R': 0.9668}

--- Qwen3-VL-8B | CoT: Without | Dataset: BrainFace | OPEN ---
    [0/17] gold='Head' pred='Head, axial CT scan'


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Metrics -- Qwen3-VL-8B (Without CoT, BrainFace): {'BLEU': 0.0586, 'ROUGE-1': 0.298, 'ROUGE-2': 0.0, 'ROUGE-L': 0.298, 'BERTScore F1': 0.8886, 'BERTScore P': 0.8769, 'BERTScore R': 0.9015}
Cleared cache for Qwen/Qwen3-VL-8B-Instruct (freed ~32.7 GB)


In [14]:
# ---------------------------------------------------------------------------
# 9. Final summary table for this DATASET
# ---------------------------------------------------------------------------
open_summary_df = pd.DataFrame(open_summary_rows)
print("\n" + "=" * 100)
print(f"OPEN-ENDED SUMMARY TABLE -- {DATASET} -- BLEU/ROUGE/BERTScore, all models, both conditions")
print("=" * 100)
print(open_summary_df.to_string(index=False))

open_summary_df.to_csv(f"{OUTPUT_DIR}/open_comparison_summary_{DATASET}.csv", index=False)
with open(f"{OUTPUT_DIR}/open_comparison_raw_results_{DATASET}.json", "w") as f:
    json.dump(all_open_results, f, indent=2)

print(f"\nSaved: {OUTPUT_DIR}/open_comparison_summary_{DATASET}.csv")
print(f"Saved: {OUTPUT_DIR}/open_comparison_raw_results_{DATASET}.json")
print("(Both will appear under this notebook's 'Output' tab / Data pane on Kaggle.)")

next_dataset = "BrainFace" if DATASET == "Neck" else "Neck"
print(f"\nNext: restart the session, set DATASET = {next_dataset!r} in section 2, and run again.")


OPEN-ENDED SUMMARY TABLE -- BrainFace -- BLEU/ROUGE/BERTScore, all models, both conditions
  Dataset    Model Name     CoT   BLEU  ROUGE-1  ROUGE-2  ROUGE-L  BERTScore F1  BERTScore P  BERTScore R
BrainFace    Gemma4-E4B    With 0.0783   0.4735   0.0131   0.4735        0.9156       0.9135       0.9187
BrainFace    Gemma4-E4B Without 0.0303   0.3353   0.0000   0.3353        0.8885       0.8771       0.9011
BrainFace Qwen2.5-VL-7B    With 0.2235   0.8882   0.1629   0.8882        0.9836       0.9836       0.9837
BrainFace Qwen2.5-VL-7B Without 0.0870   0.4510   0.0000   0.4510        0.9262       0.9295       0.9234
BrainFace     Llama-11B    With 0.0369   0.6261   0.0588   0.6261        0.9402       0.9502       0.9308
BrainFace     Llama-11B Without 0.0314   0.2353   0.0000   0.2353        0.9157       0.9227       0.9095
BrainFace   Qwen3-VL-8B    With 0.1985   0.8007   0.1307   0.8007        0.9662       0.9656       0.9668
BrainFace   Qwen3-VL-8B Without 0.0586   0.2980   0.0000   0